In [1]:
import sys
import os

# Add the src/ folder to Python's path so we can import from it
sys.path.insert(0, os.path.abspath("../src"))

print("Path configured. Ready to import.")


Path configured. Ready to import.


In [2]:
from ingestion import load_and_chunk_papers
from retrieval import get_or_build_vector_store, query, print_results
from rag_chain import build_rag_chain, ask, print_answer

print("Imports successful.")

f:\RAG-project\VisualRAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports successful.


In [3]:
# This walks data/papers/, loads every PDF, and splits into chunks
chunks = load_and_chunk_papers(papers_dir="../data/papers")

print(f"\nTotal chunks created: {len(chunks)}")
print(f"Expected range: 1000–6000 for 20–30 papers")

Found 23 PDF(s). Loading...
  Processing: A survey of deep learning methods and datasets for hand pose.pdf
  Processing: A_Review_on_3D_Hand_Pose_and_Shape_Reconstruction_from_Color_Images.pdf
  Processing: Advances in vision-based deep learning methods for interacting hands reconstruction A survey.pdf
  Processing: Best-Known-Performance-Optimizations-for-D400-Stereo-Cameras-over-Lifetime-rev-1.31.pdf
  Processing: Doosti_HOPE-Net_A_Graph-Based_Model_for_Hand-Object_Pose_Estimation_CVPR_2020_paper.pdf
  Processing: Hasson_Learning_Joint_Reconstruction_of_Hands_and_Manipulated_Objects_CVPR_2019_paper.pdf
  Processing: Jian_AffordPose_A_Large-Scale_Dataset_of_Hand-Object_Interactions_with_Affordance-Driven_Hand_ICCV_2023_paper.pdf
  Processing: Kwon_H2O_Two_Hands_Manipulating_Objects_for_First_Person_Interaction_Recognition_ICCV_2021_paper.pdf
  Processing: Liu_HOI4D_A_4D_Egocentric_Dataset_for_Category-Level_Human-Object_Interaction_CVPR_2022_paper.pdf
  Processing: Malik_HandVoxNet_De

In [4]:
# Always inspect a sample before embedding — catch bad PDFs early
sample = chunks[0]

print("=== Sample Chunk ===")
print(f"Source   : {sample.metadata['source']}")
print(f"Page     : {sample.metadata['page']}")
print(f"Char len : {len(sample.page_content)}")
print(f"\nContent preview:\n{sample.page_content[:500]}")

=== Sample Chunk ===
Source   : A survey of deep learning methods and datasets for hand pose.pdf
Page     : 0
Char len : 964

Content preview:
Computers & Graphics 116 (2023) 474–490
Contents lists available at ScienceDirect
Computers & Graphics
journal homepage: www.elsevier.com/locate/cag
Survey Paper
A survey of deep learning methods and datasets for hand pose
estimation from hand-object interaction images✩
Taeyun Woo, Wonjung Park, Woohyun Jeong, Jinah Park ∗
Korea Advanced Institution of Science and Technology, Daejeon, Republic of Korea
a r t i c l e i n f o
Article history:
Received 12 April 2023
Received in revised form 28 July


In [5]:
from collections import Counter

sources = [chunk.metadata["source"] for chunk in chunks]
counts = Counter(sources)

print(f"{'Paper':<45} {'Chunks':>6}")
print("-" * 53)
for paper, count in sorted(counts.items()):
    print(f"{paper:<45} {count:>6}")

Paper                                         Chunks
-----------------------------------------------------
A survey of deep learning methods and datasets for hand pose.pdf    162
A_Review_on_3D_Hand_Pose_and_Shape_Reconstruction_from_Color_Images.pdf     45
Advances in vision-based deep learning methods for interacting hands reconstruction A survey.pdf    114
Best-Known-Performance-Optimizations-for-D400-Stereo-Cameras-over-Lifetime-rev-1.31.pdf     38
Doosti_HOPE-Net_A_Graph-Based_Model_for_Hand-Object_Pose_Estimation_CVPR_2020_paper.pdf     53
Hasson_Learning_Joint_Reconstruction_of_Hands_and_Manipulated_Objects_CVPR_2019_paper.pdf     66
Jian_AffordPose_A_Large-Scale_Dataset_of_Hand-Object_Interactions_with_Affordance-Driven_Hand_ICCV_2023_paper.pdf     68
Kwon_H2O_Two_Hands_Manipulating_Objects_for_First_Person_Interaction_Recognition_ICCV_2021_paper.pdf     75
Liu_HOI4D_A_4D_Egocentric_Dataset_for_Category-Level_Human-Object_Interaction_CVPR_2022_paper.pdf     64
Malik_HandVoxNet_

In [6]:
# First run: embeds everything and saves to disk (~5-15 min depending on your machine)
# Subsequent runs: loads from disk in seconds
vector_store = get_or_build_vector_store(papers_dir="../data/papers")

Existing vector store found (1591 chunks). Loading...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4584.24it/s]


Loaded vector store: 1591 chunks in 'cv_papers'


In [7]:
count = vector_store._collection.count()
print(f"Chunks stored in ChromaDB: {count}")
print(f"Matches chunks created   : {len(chunks)}")
print(f"All accounted for        : {count == len(chunks)}")

Chunks stored in ChromaDB: 1591
Matches chunks created   : 1591
All accounted for        : True


In [8]:
# Build the RAG chain
chain, retriever = build_rag_chain(vector_store)  # now returns two things

test_questions = [
    "What is the mAP of Fast YOLO on PASCAL VOC 2007?",
    "What dataset does HOPE-Net use for evaluation?",
    "How does HandOccNet handle occlusion?",
    "What is the main contribution of HOI4D dataset?",
]

for q in test_questions:
    print(f"\nQ: {q}")
    result = ask(chain, retriever, q)  # now takes three arguments
    print_answer(result)


Q: What is the mAP of Fast YOLO on PASCAL VOC 2007?

ANSWER:
I don't have enough information in my paper collection to answer this. 

The provided context only mentions the mAP of YOLO on VOC 2012 test set, which is 57.9% (Yolo.pdf, Page: 6), but it does not provide the mAP of Fast YOLO on PASCAL VOC 2007.

------------------------------------------------------------
SOURCES:
------------------------------------------------------------

[1] Yolo.pdf — Page 6
    Table 3: PASCAL VOC 2012 Leaderboard. YOLO compared with the full comp4 (outside data allowed) public leaderboard as of
November 6th, 2015. Mean average precision and per-class average precision are s...

[2] Yolo.pdf — Page 0
    means we can process streaming video in real-time with
less than 25 milliseconds of latency. Furthermore, YOLO
achieves more than twice the mean average precision of
other real-time systems. For a dem...

[3] Yolo.pdf — Page 5
    of results on VOC 2007. We compare YOLO to Fast R-
CNN since Fast R-CN

In [9]:
single_question = "What tracking method does PhysTwin use?"
result = ask(chain, retriever, single_question)
print_answer(result)


ANSWER:
+1… … 
Stiffness
Simulated Geometry and MotionGaussian Rendering
Figure 2. Overview of Our PhysTwin Framework. We present an overview of our PhysTwin framework, where the core representation
includes geometry, topology, physical parameters (associated with springs and contacts), and Gaussian kernels. To optimize PhysTwin,
we minimize the rendering loss and the discrepancy between simulated and observed geometry/motion. The rendering loss optimizes the

What is the main goal of the PhysTwin framework, and what are the two stages of optimization in this framework?
The main goal of the PhysTwin framework is to minimize the discrepancy between the predicted observation and the actual observation, as stated in the phystwin.pdf (Page: 3). 
The two stages of optimization in the PhysTwin framework are: 
1. Physics and Geometry Optimization: This stage focuses on optimizing the geometry and physical parameters, as outlined in phystwin.pdf (Page: 3).
2. Appearance-related parameters opt

In [10]:
evaluation = [
    {
        "query": "What is the mAP of Fast YOLO on PASCAL VOC 2007?",
        "expected_paper": "Yolo.pdf",
        "top_result_source": None,
        "correct": None,
    },
    {
        "query": "What tracking method does PhysTwin use?",
        "expected_paper": "phystwin.pdf",
        "top_result_source": None,
        "correct": None,
    },
]

# Run each query and fill in top result automatically
for e in evaluation:
    result = ask(chain, retriever, e["query"])
    e["top_result_source"] = result["sources"][0]["source"] if result["sources"] else "None"

# Print scorecard — manually set "correct" to True/False after inspecting
for e in evaluation:
    print(f"Query    : {e['query'][:55]}")
    print(f"Expected : {e['expected_paper']}")
    print(f"Got      : {e['top_result_source']}")
    print()

Query    : What is the mAP of Fast YOLO on PASCAL VOC 2007?
Expected : Yolo.pdf
Got      : Yolo.pdf

Query    : What tracking method does PhysTwin use?
Expected : phystwin.pdf
Got      : phystwin.pdf

